In [3]:

import numpy as np 
import pandas as pd 

import os,glob,random,math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("USING DEVICE: ",device)

import kagglehub

seed=42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)



USING DEVICE:  cpu


In [4]:

import torchaudio
path="/kaggle/input/datasets/a24998667/librispeech"
print(os.listdir(path))

['test-clean', 'train-other-500', 'train-clean-100', 'BOOKS.TXT', 'README.TXT', 'test-other', 'dev-clean', 'LICENSE.TXT', 'dev-other', 'SPEAKERS.TXT', 'CHAPTERS.TXT', 'data_voip_en.tgz', 'train-clean-360']


In [10]:
train_path="/kaggle/input/datasets/a24998667/librispeech/train-clean-100/LibriSpeech/train-clean-100"
val_path="/kaggle/input/datasets/a24998667/librispeech/dev-clean/LibriSpeech/dev-clean"
print("train dataset :", len(train_path))
print("val dataset :", len(val_path))


train dataset : 88
val dataset : 76


In [11]:
import torchaudio
from torch.utils.data import Dataset

class LibriSpeechKaggleDataset(Dataset):
    def __init__(self, data_list):
        """
        data_list: A list of dicts/tuples containing 'file_path' and 'transcript'
        """
        self.data = data_list

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        item = self.data[index]
        file_path = item['file_path']
        transcript = item['transcript']
        
        # Load waveform and sample rate
        waveform, sr = torchaudio.load(file_path)
        
        return waveform, sr, transcript


In [12]:


def load_librispeech_metadata(base_dir):
    dataset_samples = []
    
    for speaker_id in os.listdir(base_dir):
        speaker_path = os.path.join(base_dir, speaker_id)
        if not os.path.isdir(speaker_path):
            continue
            
        for chapter_id in os.listdir(speaker_path):
            chapter_path = os.path.join(speaker_path, chapter_id)
            if not os.path.isdir(chapter_path):
                continue
                
            trans_file_name = f"{speaker_id}-{chapter_id}.trans.txt"
            trans_file_path = os.path.join(chapter_path, trans_file_name)
            
            if os.path.exists(trans_file_path):
                with open(trans_file_path, 'r') as f:
                    for line in f:
                        parts = line.strip().split(' ', 1)
                        if len(parts) == 2:
                            file_id, transcript = parts
                            audio_file_path = os.path.join(chapter_path, f"{file_id}.flac")
                            
                            if os.path.exists(audio_file_path):
                                dataset_samples.append({
                                    'file_path': audio_file_path,
                                    'transcript': transcript
                                })
    return dataset_samples

 
train_sample = load_librispeech_metadata(train_path)

val_sample = load_librispeech_metadata(val_path)



In [14]:
train_dataset = LibriSpeechKaggleDataset(train_sample)
val_dataset = LibriSpeechKaggleDataset(val_sample)

waveform, sr, transcript = train_dataset[0]

print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))
print(waveform,sr,transcript)

Train samples: 28539
Val samples: 2703
tensor([[-0.0037, -0.0050, -0.0040,  ...,  0.0031,  0.0016,  0.0037]]) 16000 ALL THINGS ARE CHANGES NOT INTO NOTHING BUT INTO THAT WHICH IS NOT AT PRESENT MARCUS AURELIUS


In [15]:
import random
subset_size=3000
train_indices=random.sample(range(len(train_dataset)),min(subset_size,len(train_dataset)))
val_indices=random.sample(range(len(val_dataset)),min(500,len(val_dataset)))

In [18]:
import string 
CHARS=" '" + string.ascii_lowercase
BLANK_IDX=0
char2idx={c:i + 1 for i,c in enumerate(CHARS)}
idx2char={i: c for c,i in char2idx.items()}
idx2char[BLANK_IDX]="_"

VOCAB_SIZE=len(char2idx) + 1
print("Vocab size (incl. Blank):",VOCAB_SIZE)

def text_to_label(text):
    text=text.lower()
    return [char2idx[c] for c in text if c in char2idx]

def label_to_text(label):
    return "".join(idx2char[l] for l in label if l !=BLANK_IDX)

Vocab size (incl. Blank): 29


In [ ]:
SAMPLE_RATE = 16000
N_MELS = 40
N_FFT = 400
HOP_LENGTH = 160

def hz_to_mel(hz):
    return 2595.0 * math.log10(1.0 + hz / 700.0)

def mel_to_hz(mel):
    return 700.0 * (10.0 ** (mel / 2595.0) - 1.0)

def build_mel_filterbank(sr=SAMPLE_RATE, n_fft=N_FFT, n_mels=N_MELS):
    low_mel, high_mel = hz_to_mel(0), hz_to_mel(sr / 2)
    mel_points = torch.linspace(low_mel, high_mel, n_mels + 2)
    hz_points = torch.tensor([mel_to_hz(m.item()) for m in mel_points])
    bin_points = torch.floor((n_fft + 1) * hz_points / sr).long()

    n_freq_bins = n_fft // 2 + 1
    fbank = torch.zeros(n_mels, n_freq_bins)
    for m in range(1, n_mels + 1):
        left, center, right = bin_points[m-1].item(), bin_points[m].item(), bin_points[m+1].item()
        for k in range(left, center):
            if 0 <= k < n_freq_bins:
                fbank[m-1, k] = (k - left) / max(center - left, 1)
        for k in range(center, right):
            if 0 <= k < n_freq_bins:
                fbank[m-1, k] = (right - k) / max(right - center, 1)
    return fbank

MEL_FBANK = build_mel_filterbank().to(device)
HANN_WINDOW = torch.hann_window(N_FFT).to(device)

def log_mel_spectrogram(waveform):
        pad = N_FFT // 2
    waveform = F.pad(waveform, (pad, pad), mode='reflect')
    n_frames = 1 + (waveform.shape[0] - N_FFT) // HOP_LENGTH

    frames = torch.stack([
        waveform[i*HOP_LENGTH : i*HOP_LENGTH + N_FFT]
        for i in range(n_frames)
    ])
    frames = frames * HANN_WINDOW

    spec = torch.fft.rfft(frames, dim=1)
    power_spec = spec.abs() ** 2
    mel_spec = power_spec @ MEL_FBANK.T
    return torch.log(mel_spec + 1e-6)

def frames_for_length(n_samples):
    padded_len = n_samples + N_FFT  
    return 1 + (padded_len - N_FFT) // HOP_LENGTH

In [ ]:
MAX-AUDIO_SECONDS=15

class LibriSpeechWrapper(Dataset):
    def __init__(self,base_dataset,indices):
        self.base=base_dataset
        self.indices=indices

    def __len__(self):
        return len(self.indices)

    def __getitem(self,idx):
        waveform,sr,transcript,*_=self.base[self.indices[idx]]